# Phase 3 — Numeric layer

Demonstrates the resolver, calculator, and sandbox against the live corpus — including the two failure-mode test cases (T6.2, T7.2) that prove the design. Run `01_ingestion.ipynb` first.

In [ ]:
import subprocess, sys, os
REPO_URL = "https://github.com/aditya-siraskar/STANCE.git"
if not os.path.isdir("/kaggle/working/stance"):
    subprocess.run(["git", "clone", REPO_URL, "/kaggle/working/stance"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "/kaggle/working/stance"], check=True)
sys.path.insert(0, "/kaggle/working/stance")

In [ ]:
from stance.db import get_engine
from stance.contracts import FactNotFound, UnitMismatchError
from stance.numeric.resolver import resolve_fact
from stance.numeric.calculator import build_growth_rate_spec
from stance.numeric.executor import execute_spec
from stance.numeric import sandbox
from decimal import Decimal

engine = get_engine()

## T4.1 / T4.2 — direct fact lookup and alias resolution

In [ ]:
fact = resolve_fact(engine, "revenue", "0000320193", 2024)
print(f"AAPL FY24 revenue = {fact['value']:,} {fact['unit']} [{fact['accession_no']}, fact_id={fact['fact_id']}]")

## T6.1 / T7.1 — growth-rate computation end to end

In [ ]:
current = resolve_fact(engine, "revenue", "0000320193", 2024)
prior = resolve_fact(engine, "revenue", "0000320193", 2023)
spec = build_growth_rate_spec(current["fact_id"], prior["fact_id"])
print(f"ComputationSpec: {spec.model_dump_json(indent=2)}")

growth = execute_spec(engine, spec)
print(f"\nAAPL revenue growth FY23->FY24: {growth}%")

hand_calc = (current['value'] - prior['value']) / abs(prior['value']) * 100
print(f"Hand-calculated check: {hand_calc:.2f}%")
assert abs(float(growth) - hand_calc) < 0.01

## T6.2 — missing fact fails loudly (does NOT estimate)

FY21 was never ingested (corpus is FY23-FY24 only).

In [ ]:
try:
    resolve_fact(engine, "revenue", "0000320193", 2021)
    print("FAIL: should have raised FactNotFound")
except FactNotFound as e:
    print(f"Correctly failed loudly: {e}")

## T7.2 — incompatible units rejected by the sandbox

In [ ]:
try:
    sandbox.execute("difference", Decimal("100"), Decimal("5"), "usd", "shares")
    print("FAIL: should have raised UnitMismatchError")
except UnitMismatchError as e:
    print(f"Correctly rejected: {e}")